# W03 — ML-04 Search Intelligence Data Contract

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring  
**Development month:** March 2026

This notebook covers the W03 requirements: contract framing, field roles, three verification queries, a five-feature frame, a deliberate leakage demonstration, and one named limitation.

> **Security:** This notebook expects a Colab Secret named `HF_TOKEN`. Never paste the token into a code cell or commit it to GitHub.


## 0. Setup

Run this section first. In Colab, add your Hugging Face **Read** token under the 🔑 Secrets panel with the name `HF_TOKEN` and enable notebook access.


In [18]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn

import os
import pandas as pd
import numpy as np
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. In Colab open the 🔑 Secrets panel, add a secret "
        "named HF_TOKEN, paste your Hugging Face READ token as its value, and enable notebook access."
    )

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

print("Connected to the FlyRank warehouse.")
print("Development month: March 2026")


Connected to the FlyRank warehouse.
Development month: March 2026


## 1. Unit of analysis + time window

**Unit of analysis:** One row in my lane's working feature frame represents one content item for one client in a defined reporting window.

**Time window:** I use March 2026 as the development month. I deliberately avoid the `_sample` table because the assignment notes that it is exactly the final month (June 2026), which should be treated as a sealed outcome period rather than used to develop label logic.

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring.

**Decision:** Which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?

**Output:** A transparent ranking/proxy for review priority, not a guarantee that an action will improve performance.

**Action:** A content/SEO reviewer can inspect the highest-priority pages first and choose an appropriate action.


In [19]:
# Basic schema check.
schema = con.sql(f"""
DESCRIBE SELECT *
FROM {TABLES["fact_daily"]}
""").df()

display(schema[schema["column_name"].isin([
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_data_available",
    "gsc_data_available"
])])

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
10,gsc_avg_position,DOUBLE,YES,None,None,None


## 2. Fields: feature / label / context / excluded

### Features
- **Previous-period GSC impressions:** historical visibility available before the review decision.
- **Previous-period GSC clicks:** historical search clicks available before the decision.
- **Previous-period average position:** historical search-position information available before the decision.
- **Days with impressions:** historical consistency of search visibility.
- **Content age:** metadata known before the decision, where available from the content dimension.

### Label / proxy
- **Decline proxy:** whether a page's later March impressions are at least 20% below its earlier March impressions. This is an outcome proxy for framing; it is not a guarantee that refreshing a page will reverse the decline.

### Context
- Client/content identifiers are used for grouping and joins, not as predictive features.
- Content type and other metadata can be used for interpretation/context where available.

### Excluded
- `trend_direction` and `trend_pct` are excluded from honest features because the starter documentation identifies them as label-derived.
- Future-window performance is excluded from a real predictive feature set when it occurs after the decision moment.
- Private client names, URLs, and other identifying information are excluded from published work.


In [20]:
# Confirm the content dimension has the metadata fields used below.

content_schema = con.sql(f"""
DESCRIBE SELECT *
FROM {TABLES["dim_content"]}
""").df()

display(content_schema[content_schema["column_name"].isin([
    "content_hash_id",
    "content_age_days",
    "days_since_last_update",
    "content_type"
])])

,column_name,column_type,null,key,default,extra
1,content_hash_id,VARCHAR,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


## 3. Verify it with queries (grain, counts, availability, windows)

The next three cells are the three required verification queries. They use March 2026 and keep the heavy work in DuckDB.


In [21]:
# VERIFICATION QUERY 1 — Grain
grain_check = con.sql(f"""
SELECT
    COUNT(*) AS rows_in_march,
    COUNT(DISTINCT report_date) AS distinct_dates,
    COUNT(DISTINCT client_hash_id) AS distinct_clients,
    COUNT(DISTINCT content_hash_id) AS distinct_content_items,
    COUNT(DISTINCT CONCAT(CAST(report_date AS VARCHAR), '|', client_hash_id, '|', content_hash_id))
        AS distinct_date_client_content_keys
FROM {TABLES["fact_daily"]}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""").df()

display(grain_check)
print("If rows_in_march equals distinct_date_client_content_keys, the daily fact is unique at date × client × content in this slice.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_in_march,distinct_dates,distinct_clients,distinct_content_items,distinct_date_client_content_keys
0,9841378,31,55,331437,9841378


If rows_in_march equals distinct_date_client_content_keys, the daily fact is unique at date × client × content in this slice.


In [22]:
# VERIFICATION QUERY 2 — March row count and date span
march_window = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {TABLES["fact_daily"]}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""").df()

display(march_window)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [23]:
# VERIFICATION QUERY 3 — Availability using IS TRUE
availability = con.sql(f"""
SELECT
    COUNT(*) AS total_march_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_unavailable_or_unknown_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_unavailable_or_unknown_rows
FROM {TABLES["fact_daily"]}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""").df()

display(availability)
print("Availability is checked with IS TRUE because the warehouse flags are three-valued: TRUE, FALSE, or NULL.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_march_rows,ga4_available_rows,ga4_unavailable_or_unknown_rows,gsc_available_rows,gsc_unavailable_or_unknown_rows
0,9841378,413966,9427412,3611061,6230317


Availability is checked with IS TRUE because the warehouse flags are three-valued: TRUE, FALSE, or NULL.


### Five-feature frame

For Lane 2, I use five historical signals that are available before a review decision. I aggregate them to one row per client × content for March 2026.

| Feature | Available when? |
|---|---|
| `impressions_prev_half` | Knowable from the earlier part of the March reporting window. |
| `clicks_prev_half` | Knowable from historical clicks in the earlier part of the window. |
| `avg_position_prev_half` | Knowable from historical search-position observations in the earlier part of the window. |
| `days_with_impressions_prev_half` | Knowable from the historical daily record before the decision point. |
| `content_age_days` | Knowable from content metadata before the decision point, where available. |

These are transparent review signals, not proof that a specific content action will improve a page.


In [24]:
# Build the five-feature frame and a separate outcome proxy.
# March 1–15 = historical feature window; March 16–31 = later outcome window.

feature_frame = con.sql(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM {TABLES["fact_daily"]}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
),

agg AS (
    SELECT
        client_hash_id,
        content_hash_id,

        -- Feature 1
        SUM(
            CASE
                WHEN report_date < DATE '2026-03-16'
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS impressions_prev_half,

        -- Feature 2
        SUM(
            CASE
                WHEN report_date < DATE '2026-03-16'
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS clicks_prev_half,

        -- Feature 3
        AVG(
            CASE
                WHEN report_date < DATE '2026-03-16'
                 AND gsc_avg_position > 0
                THEN gsc_avg_position
            END
        ) AS avg_position_prev_half,

        -- Feature 4
        COUNT(
            DISTINCT CASE
                WHEN report_date < DATE '2026-03-16'
                 AND COALESCE(gsc_impressions, 0) > 0
                THEN report_date
            END
        ) AS days_with_impressions_prev_half,

        -- Outcome window
        SUM(
            CASE
                WHEN report_date >= DATE '2026-03-16'
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS impressions_later_half

    FROM march
    GROUP BY 1, 2
),

content_meta AS (
    SELECT
        content_hash_id,
        ANY_VALUE(content_created_date) AS content_created_date,
        ANY_VALUE(content_updated_date) AS content_updated_date,
        ANY_VALUE(content_type) AS content_type
    FROM {TABLES["dim_content"]}
    GROUP BY content_hash_id
)

SELECT
    a.client_hash_id,
    a.content_hash_id,

    -- Five usable features
    a.impressions_prev_half,
    a.clicks_prev_half,
    a.avg_position_prev_half,
    a.days_with_impressions_prev_half,

    -- Content age calculated from the real warehouse fields
    DATE_DIFF(
        'day',
        m.content_created_date,
        DATE '2026-03-01'
    ) AS content_age_days,

    -- Outcome proxy (NOT a feature)
    a.impressions_later_half,

    CASE
        WHEN a.impressions_prev_half > 0
         AND a.impressions_later_half < 0.8 * a.impressions_prev_half
        THEN 1
        ELSE 0
    END AS is_declining_proxy

FROM agg a
LEFT JOIN content_meta m
    USING (content_hash_id)
""").df()

print(f"Feature-frame rows: {len(feature_frame):,}")
print(
    f"Declining proxy rate: "
    f"{feature_frame['is_declining_proxy'].mean():.3f}"
)

display(feature_frame.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 331,437
Declining proxy rate: 0.150


,client_hash_id,content_hash_id,impressions_prev_half,clicks_prev_half,avg_position_prev_half,days_with_impressions_prev_half,content_age_days,impressions_later_half,is_declining_proxy
0,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0.0,0.0,NaN,0,145,0.0,0
1,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0.0,0.0,NaN,0,145,0.0,0
2,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,0.0,0.0,NaN,0,145,1.0,0
3,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0.0,0.0,NaN,0,145,0.0,0
4,client_0797ff3a1fc9a6a5,content_0317b24cc1ff5c5d,0.0,0.0,NaN,0,154,0.0,0
5,client_0797ff3a1fc9a6a5,content_044c54ec4adcc4b2,0.0,0.0,NaN,0,145,0.0,0
6,client_0797ff3a1fc9a6a5,content_07573a1cc2034981,0.0,0.0,NaN,0,145,0.0,0
7,client_0797ff3a1fc9a6a5,content_084680e7da2a2ff9,0.0,0.0,NaN,0,145,0.0,0
8,client_0797ff3a1fc9a6a5,content_0c3410828632f110,0.0,0.0,NaN,0,145,0.0,0
9,client_0797ff3a1fc9a6a5,content_0e62212f207dde7a,0.0,0.0,NaN,0,145,0.0,0


### The trap: deliberate leakage experiment

I deliberately add the later-period outcome (`impressions_later_half`) as a feature. This is information that would not be available at the decision moment. The quick score should become suspiciously strong because the feature contains the outcome itself or a direct ingredient of it.

I then remove that leaked feature and retain the honest five-feature frame.


In [25]:
from sklearn.tree import DecisionTreeClassifier

honest_features = [
    "impressions_prev_half",
    "clicks_prev_half",
    "avg_position_prev_half",
    "days_with_impressions_prev_half",
    "content_age_days"
]

model_df = feature_frame.dropna(
    subset=honest_features + ["is_declining_proxy"]
).copy()

X_honest = model_df[honest_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = model_df["is_declining_proxy"].astype(int)

honest_model = DecisionTreeClassifier(
    max_depth=2, class_weight="balanced", random_state=42
)
honest_model.fit(X_honest, y)
honest_accuracy = honest_model.score(X_honest, y)

leaky_features = honest_features + ["impressions_later_half"]
X_leaky = model_df[leaky_features].replace([np.inf, -np.inf], np.nan).fillna(0)

leaky_model = DecisionTreeClassifier(
    max_depth=2, class_weight="balanced", random_state=42
)
leaky_model.fit(X_leaky, y)
leaky_accuracy = leaky_model.score(X_leaky, y)

print(f"Honest feature-set training accuracy: {honest_accuracy:.3f}")
print(f"Leaky feature-set training accuracy:  {leaky_accuracy:.3f}")
print(f"Change caused by leakage:              {leaky_accuracy - honest_accuracy:+.3f}")
print("\nLeaked feature intentionally used: impressions_later_half")
print("Honest feature set retained:", honest_features)


Honest feature-set training accuracy: 0.511
Leaky feature-set training accuracy:  0.752
Change caused by leakage:              +0.242

Leaked feature intentionally used: impressions_later_half
Honest feature set retained: ['impressions_prev_half', 'clicks_prev_half', 'avg_position_prev_half', 'days_with_impressions_prev_half', 'content_age_days']


### Leakage conclusion

The leakage experiment is deliberately invalid as a predictive setup because `impressions_later_half` is part of the outcome period. If the leaky score is higher, that is evidence that outcome information makes the task artificially easy, not evidence of a better model.

I therefore remove the leaked column and keep only the five historical features in the honest feature frame.


## 4. Data limits

**Named limitation: unbalanced and incomplete history.** The warehouse is an unbalanced panel: clients have different history depths. In addition, early rows can be GSC-only, while GA4 availability can be TRUE, FALSE, or NULL. Therefore a missing or unavailable metric should not automatically be interpreted as zero performance.

A second important limitation is that this analysis is observational. A page can be ranked as a strong candidate for review, but the ranking cannot prove that refreshing, expanding, protecting, pruning, or monitoring the page will cause future performance to improve. The output is decision support, not a causal guarantee.


In [26]:
# Final self-check numbers generated from the actual warehouse run.
print("=== W03 SELF-CHECK ===")
print("Lane: Lane 2 — Refresh / Content Opportunity Scoring")
print("Development month: March 2026")
print("Feature count: 5")
print("Leak column removed: impressions_later_half")
print("Feature-frame rows:", len(feature_frame))
print("Honest model score:", round(honest_accuracy, 3))
print("Leaky model score:", round(leaky_accuracy, 3))
print("No token is stored in this notebook code.")


=== W03 SELF-CHECK ===
Lane: Lane 2 — Refresh / Content Opportunity Scoring
Development month: March 2026
Feature count: 5
Leak column removed: impressions_later_half
Feature-frame rows: 331437
Honest model score: 0.511
Leaky model score: 0.752
No token is stored in this notebook code.


## Submission checklist

- [x] Lane 2 is explicitly stated.
- [x] Unit of analysis and March 2026 development window are stated.
- [x] Feature / label / context / excluded fields are explained.
- [x] Three verification queries are included with visible outputs.
- [x] Availability is checked using `IS TRUE`.
- [x] Five features are built and each has a decision-time rationale.
- [x] A label/outcome-derived feature is deliberately added, compared, and removed.
- [x] One named limitation is stated.
- [ ] Run **Runtime → Run all** with no errors.
- [ ] Save the executed notebook to `work/notebooks/w03_data_contract.ipynb` in your public GitHub repository.
- [ ] Submit your repository URL on the assignment card.
